Question 1 : Define Data Quality in the context of ETL pipelines. Why is it more than just data cleaning?

Answer:

Data Quality in ETL pipelines refers to the degree to which data is accurate, complete, consistent, timely, valid, and unique — making it fit for its intended use in analysis and decision-making.
Why it's more than just data cleaning:

Data cleaning is only one reactive step (fixing errors after they appear). Data Quality is a broader, proactive discipline.

Question 2 : Explain why poor data quality leads to misleading dashboards and incorrect decisions.

Answer:

1. NULL values distort aggregations


*   Txn_206 has Txn_Amount = NULL. If you calculate total revenue, this record is silently ignored → revenue appears lower than it is.

2. Duplicate records inflate metrics


*   Txn_201, 203, and 208 are all identical (C101, P11, ₹4000, 2025-12-01). A dashboard summing revenue would count ₹4000 three times instead of once → inflated sales figures.

3. Invalid/placeholder values corrupt filters


*  Txn_206 has Customer_Name = "N/A". A customer segmentation report would treat "N/A" as an actual customer → wrong customer counts.

4. Missing dates break time-series charts



*   Txn_207 has Txn_Date = NULL. It cannot appear on any date-based dashboard → transaction silently disappears from trend analysis.









Question 3 : What is duplicate data? Explain three causes in ETL pipelines.

Answer:

Duplicate data means two or more records in a dataset represent the same real-world event or entity, causing that fact to be counted more than once.

In our dataset, Txn_IDs 201, 203, and 208 are all duplicates of the same transaction (C101, P11, ₹4000, 2025-12-01).

Three causes in ETL pipelines:

Cause 1 — Re-running ETL jobs without deduplication
If an ETL job fails midway and is re-run, the already-inserted records get inserted again. Without a deduplication check, every re-run adds copies.

Cause 2 — Multiple source systems sending the same data
In enterprise systems, the same transaction may exist in both an ERP and a CRM. If both are ingested without cross-source deduplication, the same event enters twice.

Cause 3 — Merge/append without checking existing records
When new data files are appended to a data warehouse table (e.g., daily CSV loads), if there's no INSERT IGNORE or MERGE logic, overlapping date ranges get loaded multiple times.



Question 4 : Differentiate between exact, partial, and fuzzy duplicates.

Answer:



*   Exact Duplicate :- Every field is 100% identical across two or more rows.

example: Txn_201, 203, 208 — same Customer_ID, Product_ID, Quantity, Amount, Date, City



*   Partial Duplicate:- The business key fields match, but some non-key fields differ (e.g., different Txn_ID assigned)

Example: Same as above but with different Txn_IDs (201 vs 203 vs 208) — same transaction, different surrogate keys



*   Fuzzy Duplicate:-
Records refer to the same entity but with slight variations due to typos, formatting, or abbreviations

Example: "Rahul Mehta" vs "R. Mehta" vs "rahul mehta" — same person, different representations





Question 5 : Why should data validation be performed during transformation rather than after loading?

Answer:
 Because fixing errors after loading is far more costly, risky, and sometimes irreversible.





Question 6 : Explain how business rules help in validating data accuracy. Give an example.

Answer:
Business rules are domain-specific constraints that define what "correct" data looks like — beyond just syntactic correctness.

Types of business rule validations:



*   Range check

Example: Txn_Amount > 0



* Null check

Example: Quantity must not be NULL for a completed sale



* Referential check

Exaample: Customer_ID must exist in Customers_Master



*   Format check

Example: Txn_Date must follow YYYY-MM-DD



*   Cross-field check

Example: Txn_Amount = Quantity × Unit_Price














Question 7 : Write an SQL query on Sales_Transactions to list all duplicate keys and their counts using the
business key (Customer_ID + Product_ID + Txn_Date + Txn_Amount )

Answer:



In [ ]:
SELECT
    Customer_ID,
    Product_ID,
    Txn_Date,
    Txn_Amount,
    COUNT(*) AS duplicate_count
FROM
    Sales_Transactions
GROUP BY
    Customer_ID,
    Product_ID,
    Txn_Date,
    Txn_Amount
HAVING
    COUNT(*) > 1;

Answer 8



* Customers_Master has Customer_IDs: C101, C102, C103, C104


*  Sales_Transactions has Customer_IDs: C101, C102, C103, C104, C105, C106






In [ ]:
SELECT DISTINCT
    st.Customer_ID,
    st.Customer_Name,
    st.Txn_ID
FROM
    Sales_Transactions st
LEFT JOIN
    Customers_Master cm
    ON st.Customer_ID = cm.CustomerID
WHERE
    cm.CustomerID IS NULL;

Explaination





*   A LEFT JOIN keeps all rows from Sales_Transactions, even those with no match in Customers_Master.

*   WHERE cm.CustomerID IS NULL isolates rows where no matching master record was found — these are the integrity violations.

*   C105 (Txn_206) and C106 (Txn_207) do not exist in Customers_Master → they are orphan transactions that violate referential integrity

